In [7]:
DEBUG = False
EVAL_SIZE = 100 if DEBUG else 1000

RETRIEVAL_DATASET = "/kaggle/input/datasets/keyaaness/cost-aware-adaptive-rag-retrieval-v1/retrieval_artifacts"
OUTPUT_DIR = "/kaggle/working/fixed_rag_results_v2"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

RETRIEVAL_K = 5
MAX_NEW_TOKENS = 64
SEED = 42

!pip install -q sentence-transformers transformers accelerate pandas numpy pyarrow faiss-cpu

import os
import re
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [8]:
corpus = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "corpus.parquet")
)

final_test = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "final_test.parquet")
)

faiss_index = faiss.read_index(
    os.path.join(RETRIEVAL_DATASET, "faiss.index")
)

eval_df = final_test.sample(
    n=EVAL_SIZE,
    random_state=SEED
).reset_index(drop=True)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL,
    use_fast=True
)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

generator = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=dtype
)

generator.to(device)
generator.eval()

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token = generator_tokenizer.eos_token

print("Corpus:", corpus.shape)
print("Evaluation examples:", len(eval_df))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Corpus: (62933, 5)
Evaluation examples: 1000


In [9]:
def dense_retrieve(query, k=5):
    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    scores, indices = faiss_index.search(
        embedding,
        k
    )

    documents = []

    for score, idx in zip(scores[0], indices[0]):
        row = corpus.iloc[int(idx)]
        documents.append({
            "doc_id": int(row["doc_id"]),
            "title": str(row["title"]),
            "text": str(row["text"]),
            "score": float(score)
        })

    return documents


def build_context(documents):
    return "\n\n".join(
        f"[Document {i}]\nTitle: {doc['title']}\n{doc['text']}"
        for i, doc in enumerate(documents, 1)
    )


def generate_answer(question, context):
    messages = [
        {
            "role": "system",
            "content": "Answer using only the provided context. Return only the concise answer."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
        }
    ]

    prompt = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = generator.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=generator_tokenizer.pad_token_id,
            eos_token_id=generator_tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    answer = generator_tokenizer.decode(
        output[0][prompt_length:],
        skip_special_tokens=True
    ).strip()

    return answer, elapsed


def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def exact_match(prediction, reference):
    return float(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1(prediction, reference):
    p = normalize_answer(prediction).split()
    r = normalize_answer(reference).split()

    if not p and not r:
        return 1.0

    if not p or not r:
        return 0.0

    pc = {}
    rc = {}

    for token in p:
        pc[token] = pc.get(token, 0) + 1

    for token in r:
        rc[token] = rc.get(token, 0) + 1

    overlap = sum(
        min(pc[token], rc.get(token, 0))
        for token in pc
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(r)

    return 2 * precision * recall / (precision + recall)


def count_tokens(text):
    return len(
        generator_tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

In [10]:
results = []

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Fixed Dense@5 RAG"
):
    query = str(row["question"])
    gold = str(row["answer"])

    retrieval_start = time.perf_counter()

    documents = dense_retrieve(
        query,
        RETRIEVAL_K
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    retrieval_time = time.perf_counter() - retrieval_start

    context = build_context(documents)

    prediction, generation_time = generate_answer(
        query,
        context
    )

    results.append({
        "id": str(row["id"]),
        "question": query,
        "gold_answer": gold,
        "prediction": prediction,
        "retrieved_doc_ids": json.dumps(
            [d["doc_id"] for d in documents]
        ),
        "context_tokens": count_tokens(context),
        "generation_tokens": len(
            generator_tokenizer.encode(
                prediction,
                add_special_tokens=False
            )
        ),
        "retrieval_time_ms": retrieval_time * 1000,
        "generation_time_ms": generation_time * 1000,
        "total_time_ms": (
            retrieval_time + generation_time
        ) * 1000,
        "exact_match": exact_match(prediction, gold),
        "f1": token_f1(prediction, gold)
    })

results_df = pd.DataFrame(results)

summary = {
    "num_examples": len(results_df),
    "exact_match": float(results_df["exact_match"].mean()),
    "f1": float(results_df["f1"].mean()),
    "avg_context_tokens": float(results_df["context_tokens"].mean()),
    "avg_generation_tokens": float(results_df["generation_tokens"].mean()),
    "avg_retrieval_time_ms": float(results_df["retrieval_time_ms"].mean()),
    "avg_generation_time_ms": float(results_df["generation_time_ms"].mean()),
    "avg_total_time_ms": float(results_df["total_time_ms"].mean()),
    "median_total_time_ms": float(results_df["total_time_ms"].median()),
    "p95_total_time_ms": float(results_df["total_time_ms"].quantile(0.95)),
    "retrieval_depth": 5
}

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_path / "fixed_rag_results.csv",
    index=False
)

with open(output_path / "fixed_rag_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

results_df[
    [
        "id",
        "question",
        "gold_answer",
        "prediction",
        "exact_match",
        "f1",
        "context_tokens",
        "total_time_ms"
    ]
].head(30).to_csv(
    output_path / "fixed_rag_examples.csv",
    index=False
)

print(pd.Series(summary).to_string())

Fixed Dense@5 RAG:   0%|          | 0/1000 [00:00<?, ?it/s]

num_examples              1000.000000
exact_match                  0.294000
f1                           0.437735
avg_context_tokens         666.994000
avg_generation_tokens        8.232000
avg_retrieval_time_ms       22.761879
avg_generation_time_ms     481.260191
avg_total_time_ms          504.022069
median_total_time_ms       381.043305
p95_total_time_ms         1329.130250
retrieval_depth              5.000000
